# 14 Dashboard: model selection
Numbers for the **How the model was chosen** section of the dashboard's Model page: the models compared, the three that were tuned, and CatBoost's tuning steps.

They come from the **saved outputs of `04_modeling.ipynb`**, so nothing is retrained (re-running `04` takes hours). If `04` is run again, run this notebook after it to pick up the new results.

- Tables: VS Code saves a full copy of each displayed DataFrame with the output (`application/vnd.microsoft.datawrangler.viewer.v0+json`); that copy is read back here.
- Best parameters: read from the text the search cell printed.

In [1]:
import ast
import json
import re
import pandas as pd
from carpricepredictor.shared import BEST_PARAMS_PATH, DASHBOARD_DIR

cells = [c for c in json.load(open("04_modeling.ipynb"))["cells"] if c["cell_type"] == "code"]

def cell(starts_with):
    """The 04 code cell whose source starts with this text."""
    return next(c for c in cells if "".join(c["source"]).lstrip().startswith(starts_with))

def table(c):
    """The DataFrame a 04 cell displayed, from the copy VS Code saved with its output."""
    data = next(o["data"] for o in c["outputs"] if o["output_type"] == "execute_result")
    view = data["application/vnd.microsoft.datawrangler.viewer.v0+json"]
    df = pd.DataFrame(view["rows"], columns=[col["name"] for col in view["columns"]])
    for col in view["columns"]:
        if col["type"] in ("float", "integer"):
            df[col["name"]] = pd.to_numeric(df[col["name"]])
    return df.drop(columns="index")

## Models compared
5-fold CV of every model with default settings (`04`, "Training models to test"), best first.

In [2]:
compared = table(cell("cv = KFold"))
compared

,model,rmse_log,mae_log,r2
0,CatBoostRegressor,0.338422,0.213232,0.850245
1,HistGradientBoostingRegressor,0.357840,0.231266,0.832568
2,RandomForestRegressor,0.363325,0.230392,0.827380
3,Ridge,0.413085,0.273508,0.776885
4,LinearRegression,0.413103,0.273473,0.776866
5,KNeighborsRegressor,0.461951,0.331319,0.720976
6,DecisionTreeRegressor,0.510314,0.327116,0.659502
7,DummyRegressor,0.875559,0.707415,-0.002325


## The top 3, tuned
`RandomizedSearchCV` on the three best (`04`, "Hyperparameter tuning"): CV score with default settings vs tuned, and the best parameters found. The parameters are parsed from the printed lines (`name score {params}`); `np.float64(x)` is turned back into a plain number first.

In [3]:
tuned = table(cell("baseline = results"))

printed = "".join(next(o["text"] for o in cell("searches = {}")["outputs"] if o["output_type"] == "stream"))
best = {}
for line in printed.strip().splitlines():
    name, score, params = line.split(" ", 2)
    params = ast.literal_eval(re.sub(r"np\.float64\(([^)]*)\)", r"\1", params))
    best[name] = ", ".join(f"{k.removeprefix('reg__')}={round(v, 4) if isinstance(v, float) else v}" for k, v in params.items())
tuned["best_params"] = tuned["model"].map(best)
tuned

,model,rmse_default,rmse_tuned,gain_vs_default,gain_vs_linear,best_params
0,CatBoostRegressor,0.338422,0.312849,0.025573,0.100253,"depth=10, iterations=1500, l2_leaf_reg=2.8581,..."
1,HistGradientBoostingRegressor,0.357840,0.323463,0.034377,0.089639,"l2_regularization=0.0292, learning_rate=0.0688..."
2,RandomForestRegressor,0.363325,0.351705,0.011620,0.061398,"n_estimators=200, min_samples_leaf=1, max_feat..."


## CatBoost's path
From the linear baseline to the final parameters (`04`, "CatBoost deep tuning"). The "native" step tried CatBoost's own handling of the categories; it wasn't better, so one-hot encoding was kept. `best_params.json` says which one was used.

In [4]:
steps = table(cell('pd.DataFrame({\n    "step"'))
use_native = json.loads(BEST_PARAMS_PATH.read_text())["use_native"]
steps["used"] = ~steps["step"].str.contains("native") | use_native
steps

,step,rmse_log,used
0,LinearRegression (baseline),0.413103,True
1,CatBoost default,0.338422,True
2,CatBoost tuned (round 1),0.312849,True
3,"CatBoost native, round-1 params",0.313486,False
4,CatBoost tuned (round 2),0.310499,True


## Save

In [5]:
DASHBOARD_DIR.mkdir(exist_ok=True)
out = {"ms_compared.csv": compared, "ms_tuned.csv": tuned, "ms_catboost_steps.csv": steps}
for name, t in out.items():
    t.to_csv(DASHBOARD_DIR / name, index=False)
    print(f"{name:22} {len(t):>3} rows")

ms_compared.csv          8 rows
ms_tuned.csv             3 rows
ms_catboost_steps.csv    5 rows
